In [1]:
from rdflib import Graph
from rdfine import GraphReader
from compilers import (
    PipelineGenerator,
    ProjectBuilder,
    LdioConfigCompiler)

#### Loading the graph

In [2]:
# Loading the graph
input_folder = "..\\data\\"
catalog_graph = Graph()
catalog_graph.parse(input_folder + "catalog.ttl", publicID = "file:///workspace/pipeline/")
catalog_reader = GraphReader(catalog_graph)
catalog_reader = catalog_reader.infer(input_folder + "inference_rules.yaml")

#### Compiling the pipeline build

`PipelineGenerator` orchestrates the full chain. It runs `PipelineExtractor` first (the only compiler that needs the `pipeline_id`, and it seeds the `tcs:PipelineBuild` node), then loops over `Compiler._registry`, invoking every compiler whose `applies_to` trigger becomes true against the growing build graph. Execution order emerges from those triggers rather than from any class-level rank.

In [3]:
pipeline_id = ":DemonstratorPipeline"
gen = PipelineGenerator(pipeline_id, catalog_reader.graph)
build_graph = gen.compile()

# Which compilers actually ran?
[cls.__name__ for cls in gen.compilers]

['PipelineExtractor',
 'PipelineAssembler',
 'SemanticWorksCompiler',
 'LdioConfigCompiler',
 'RdfcConfigCompiler',
 'DockerComposeCompiler']

#### Inspecting the compiled files

Every compiler that produces a file attaches it to the `tcs:PipelineBuild` as a `tcs:File` node via `tcs:compiledFile`. The build graph is now self-describing: it knows which files should be written, where, and with what content.

`ProjectBuilder` collects those nodes into a DataFrame on `builder.files` for inspection before any IO happens.

In [4]:
builder = ProjectBuilder(build_graph)

for _, row in builder.files.iterrows():
    print(f"=== {row['filepath']}/{row['filename']} ===")
    print(row['content'])
    print()

=== ldio/config.yml ===
input:
  adapter:
    name: Ldio:RdfAdapter
  name: Ldio:HttpInPoller
  config:
    cron: '*/10 * * * * *'
    url: https://dishacled-api.azurewebsites.net/api/v1/source-a/current
transformers:
- name: Ldio:SparqlConstructTransformer
  config:
    query: "\r\n            PREFIX dct: <http://purl.org/dc/terms/> .\r\n        \
      \    PREFIX prov: <http://www.w3.org/ns/prov#> .\r\n            CONSTRUCT {\r\
      \n              ?versionedS ?p ?o ;\r\n                  prov:generatedAtTime\
      \ ?generatedAtTime ;\r\n                  dct:isVersionOf ?s .\r\n         \
      \   } WHERE {\r\n              ?s ?p ?o .\r\n              BIND(URI(CONCAT(STR(?s),\
      \ '/', STR(?now))) as ?versionedS)\r\n              BIND (NOW() as ?generatedAtTime)\r\
      \n            }\r\n          "
outputs:
- name: Ldio:HttpOut


=== rdfc/pipeline.ttl ===
@base <file:///workspace/pipeline/> .
@prefix : <http://example.org/example/> .
@prefix ns1: <https://no_prefix_avai

#### Writing the project to disk

`ProjectBuilder.write(target_dir)` materializes every collected file under the given directory, creating parent folders as needed. Existing files at the same path are overwritten. The call returns the absolute paths it wrote.

In [5]:
written = builder.write("../out/demonstrator")
for path in written:
    print(path)

C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\demonstrator\ldio\config.yml
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\demonstrator\rdfc\pipeline.ttl
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\demonstrator\docker-compose.yml


#### Inspecting compiler internals

`PipelineGenerator` keeps the compiler instances it ran on `gen.compilers`, keyed by class. Each instance retains its intermediate state — useful for debugging when an output doesn't look right.

In [6]:
gen.compilers[LdioConfigCompiler].df_steps

,component,type,name,config
0,ldio:SparqlConstructTransformer,Transformer,Ldio:SparqlConstructTransformer,:config_2
1,ldio:RdfAdapter,Adapter,Ldio:RdfAdapter,NaN
2,ldio:HttpInPoller,Input,Ldio:HttpInPoller,:config_6
3,ldio:HttpOut,Output,Ldio:HttpOut,NaN
